In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
dataset = pd.read_csv('Mall_Customers.csv')

In [3]:
dataset

,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40
...,...,...,...,...,...
195,196,Female,35,120,79
196,197,Female,45,126,28
197,198,Male,32,126,74
198,199,Male,32,137,18


In [4]:
# Restore the features for all 200 customers

X = dataset.iloc[:, [3, 4]].values

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Shape:", X_scaled.shape)
print("First five scaled rows:")
print(X_scaled[:5])

Shape: (200, 2)
First five scaled rows:
[[-1.73899919 -0.43480148]
 [-1.73899919  1.19570407]
 [-1.70082976 -1.71591298]
 [-1.70082976  1.04041783]
 [-1.66266033 -0.39597992]]


In [5]:
# Create and train the model

In [7]:
#eps - Maximum distance between neighbouring points
#min_samples - Minimum points within that neighbourhood, including the point itself, for a core point

In [11]:
from sklearn.cluster import DBSCAN

DB = DBSCAN(
    eps=0.3,
    min_samples=5
)

y_DB = DB.fit_predict(X_scaled)

In [12]:
print("Cluster labels:")
print(y_DB)

print("\nNumber of clusters:", len(np.unique(y_DB)))

Cluster labels:
[ 2  0  1  0  2  0  1 -1  1  0 -1 -1 -1  0  1  0  2  0  2 -1  2  0  1  0
 -1 -1  2 -1  2 -1 -1  0 -1 -1 -1 -1 -1  0 -1  0  3 -1  3  3  3  3  3  3
  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3
  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3
  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3
  3  3  3  4  3  4  3  4  5  4  5  4  3  4  5  4  5  4  5  4  5  4  3  4
  5  4  3  4  5  4  5  4  5  4  5  4  5  4  5  4  3  4  5  4  6  4  6  4
  6 -1  6  4  6  4  6  4  6  4  6  4 -1  4  6  4 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1]

Number of clusters: 8


In [ ]:
# Count clusters and noise correctly

In [13]:
# Exclude the noise label when counting clusters
cluster_ids = sorted(set(y_DB) - {-1})

n_clusters = len(cluster_ids)
n_noise = np.sum(y_DB == -1)

print("Number of clusters:", n_clusters)
print("Noise customers:", n_noise)
print("Clustered customers:", len(y_DB) - n_noise)
print("Total customers:", len(y_DB))

Number of clusters: 7
Noise customers: 35
Clustered customers: 165
Total customers: 200


In [11]:
#Add the results to the dataset
#Labels 0, 1, 2, etc. identify clusters. Label -1 identifies customers classified as noise under the selected settings.

In [14]:
dataset["DBSCAN_Cluster"] = y_DB

print("Customers:", len(dataset))
print("Cluster labels:", len(y_DB))

display(dataset.head(10))

Customers: 200
Cluster labels: 200


,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100),DBSCAN_Cluster
0,1,Male,19,15,39,2
1,2,Male,21,15,81,0
2,3,Female,20,16,6,1
3,4,Female,23,16,77,0
4,5,Female,31,17,40,2
5,6,Female,22,17,76,0
6,7,Female,35,18,6,1
7,8,Female,23,18,94,-1
8,9,Male,64,19,3,1
9,10,Female,30,19,72,0


In [13]:
#Summarize each group
#If row -1 appears, it summarizes all noise customers collectively; they do not form a single discovered cluster.

In [15]:
summary = dataset.groupby("DBSCAN_Cluster").agg(
    Customers=("CustomerID", "count"),
    Average_Income=("Annual Income (k$)", "mean"),
    Average_Spending=("Spending Score (1-100)", "mean")
).round(2)

display(summary)

,Customers,Average_Income,Average_Spending
DBSCAN_Cluster,,,
-1,35,70.60,51.66
0,12,23.17,74.58
1,5,19.60,6.60
2,7,22.43,34.43
3,88,55.23,48.58
4,30,79.53,83.13
5,14,75.93,10.07
6,9,88.78,17.00


In [ ]:
#To inspect those customers
#Noise does not automatically mean an error or suspicious customer—
#it means the customer was not assigned to a dense group with these features and settings.

In [16]:
noise_customers = dataset[
    dataset["DBSCAN_Cluster"] == -1
]

display(noise_customers)

,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100),DBSCAN_Cluster
7,8,Female,23,18,94,-1
10,11,Male,67,19,14,-1
11,12,Female,35,19,99,-1
12,13,Female,58,20,15,-1
19,20,Female,35,23,98,-1
24,25,Female,54,28,14,-1
25,26,Male,29,28,82,-1
27,28,Male,35,28,61,-1
29,30,Female,23,29,87,-1
30,31,Male,60,30,4,-1


In [21]:
#8. Export the results

In [20]:
dataset.to_csv(
    "DBSCAN_Customer_Results.csv",
    index=False
)

summary.to_csv("DBSCAN_Cluster_Summary.csv")

# Run Step 7 before exporting this table
comparison.to_csv(
    "DBSCAN_Parameter_Comparison.csv",
    index=False
)

print("Results saved in the notebook's working folder.")

Results saved in the notebook's working folder.
